In [18]:
import pandas as pd

In [19]:
df = pd.read_excel("data/business_data.xlsx")

In [20]:
print("BUSINESS DATA")
print(df)

BUSINESS DATA
       Business Name  Category       City      State PIN Code  Latitude  \
0           ABC Cafe      Cafe  Bengaluru  Karnataka   560001   12.9716   
1      City Hospital  Hospital  Mangaluru  Karnataka   575001   12.9141   
2    Mysore Pharmacy  Pharmacy     Mysuru  Karnataka   570001   12.2958   
3   Mangalore Bakery    Bakery  Mangaluru  Karnataka   575001   12.9145   
4  Bengaluru Library   Library  Bengaluru  Karnataka   560034   12.9352   
5       XYZ Pharmacy  Pharmacy     Mysuru  Karnataka   5700XX       NaN   
6           ABC Cafe      Cafe  Bengaluru  Karnataka   560001       NaN   

   Longitude Opening Hours  
0    77.5946  9 AM - 10 PM  
1    74.8560      24 Hours  
2    76.6394   8 AM - 9 PM  
3    74.8564   8 AM - 8 PM  
4    77.6245  10 AM - 6 PM  
5        NaN           NaN  
6        NaN           NaN  


In [21]:
duplicates = df[df.duplicated(subset=["Business Name"], keep=False)]

In [22]:
print("\nDUPLICATE BUSINESSES")
print(duplicates)


DUPLICATE BUSINESSES
  Business Name Category       City      State PIN Code  Latitude  Longitude  \
0      ABC Cafe     Cafe  Bengaluru  Karnataka   560001   12.9716    77.5946   
6      ABC Cafe     Cafe  Bengaluru  Karnataka   560001       NaN        NaN   

  Opening Hours  
0  9 AM - 10 PM  
6           NaN  


In [23]:
df["PIN Code"] = df["PIN Code"].astype(str)

invalid_pins = df[
    ~df["PIN Code"].str.match(r"^\d{6}$")
]

In [24]:
print("\nINVALID PIN CODES")
print(invalid_pins)


INVALID PIN CODES
  Business Name  Category    City      State PIN Code  Latitude  Longitude  \
5  XYZ Pharmacy  Pharmacy  Mysuru  Karnataka   5700XX       NaN        NaN   

  Opening Hours  
5           NaN  


In [25]:
missing_values = df[df.isnull().any(axis=1)]

In [26]:
print("\nRECORDS WITH MISSING VALUES")
print(missing_values)


RECORDS WITH MISSING VALUES
  Business Name  Category       City      State PIN Code  Latitude  Longitude  \
5  XYZ Pharmacy  Pharmacy     Mysuru  Karnataka   5700XX       NaN        NaN   
6      ABC Cafe      Cafe  Bengaluru  Karnataka   560001       NaN        NaN   

  Opening Hours  
5           NaN  
6           NaN  


In [27]:
print("====================================")
print("   DATA VALIDATION SUMMARY")
print("====================================")

print("Total Businesses:", len(df))
print("Duplicate Records:", len(duplicates))
print("Invalid PIN Codes:", len(invalid_pins))
print("Records with Missing Values:", len(missing_values))

print("====================================")

   DATA VALIDATION SUMMARY
Total Businesses: 7
Duplicate Records: 2
Invalid PIN Codes: 1
Records with Missing Values: 2


In [28]:
# Show exactly what needs to be reviewed

print("BUSINESS RECORDS REQUIRING REVIEW")
print("=" * 50)

for index, row in df.iterrows():

    problems = []

    # Check PIN code
    if not row["PIN Code"].isdigit() or len(row["PIN Code"]) != 6:
        problems.append("Invalid PIN Code")

    # Check missing values
    if pd.isna(row["Latitude"]):
        problems.append("Missing Latitude")

    if pd.isna(row["Longitude"]):
        problems.append("Missing Longitude")

    if pd.isna(row["Opening Hours"]):
        problems.append("Missing Opening Hours")

    # Print the problem
    if problems:
        print("\nBusiness:", row["Business Name"])
        print("Problems:", ", ".join(problems))

BUSINESS RECORDS REQUIRING REVIEW

Business: XYZ Pharmacy
Problems: Invalid PIN Code, Missing Latitude, Missing Longitude, Missing Opening Hours

Business: ABC Cafe
Problems: Missing Latitude, Missing Longitude, Missing Opening Hours


In [29]:
# Calculate Data Quality Score

checks = [
    "Business Name",
    "Category",
    "City",
    "State",
    "PIN Code",
    "Latitude",
    "Longitude",
    "Opening Hours"
]

# Start with a decimal score
df["Data Quality Score"] = 100.0

# Deduct points for missing values
for column in checks:
    df.loc[df[column].isna(), "Data Quality Score"] -= 12.5

# Deduct points for invalid PIN codes
df.loc[
    ~df["PIN Code"].astype(str).str.match(r"^\d{6}$"),
    "Data Quality Score"
] -= 12.5

# Deduct points for duplicate businesses
df.loc[
    df.duplicated(subset=["Business Name"], keep=False),
    "Data Quality Score"
] -= 10

# Don't allow the score to go below 0
df["Data Quality Score"] = df["Data Quality Score"].clip(lower=0)

# Create validation status
df["Validation Status"] = df["Data Quality Score"].apply(
    lambda score: "VALID" if score == 100 else "REVIEW"
)

# Display the result
print("DATA QUALITY RESULTS")
print("=" * 60)

print(
    df[
        ["Business Name", "Data Quality Score", "Validation Status"]
    ].to_string(index=False)
)

DATA QUALITY RESULTS
    Business Name  Data Quality Score Validation Status
         ABC Cafe                90.0            REVIEW
    City Hospital               100.0             VALID
  Mysore Pharmacy               100.0             VALID
 Mangalore Bakery               100.0             VALID
Bengaluru Library               100.0             VALID
     XYZ Pharmacy                50.0            REVIEW
         ABC Cafe                52.5            REVIEW


In [30]:
# Create a column explaining why a record needs review

def find_issues(row):

    issues = []

    # PIN code check
    pin = str(row["PIN Code"])

    if not pin.isdigit() or len(pin) != 6:
        issues.append("Invalid PIN Code")

    # Missing information
    if pd.isna(row["Latitude"]):
        issues.append("Missing Latitude")

    if pd.isna(row["Longitude"]):
        issues.append("Missing Longitude")

    if pd.isna(row["Opening Hours"]):
        issues.append("Missing Opening Hours")

    # Duplicate check
    if df["Business Name"].tolist().count(row["Business Name"]) > 1:
        issues.append("Duplicate Business")

    if len(issues) == 0:
        return "No Issues"

    return ", ".join(issues)


df["Issues Found"] = df.apply(find_issues, axis=1)

print("FINAL VALIDATION REPORT")
print("=" * 100)

print(
    df[
        [
            "Business Name",
            "Data Quality Score",
            "Validation Status",
            "Issues Found"
        ]
    ].to_string(index=False)
)

FINAL VALIDATION REPORT
    Business Name  Data Quality Score Validation Status                                                                   Issues Found
         ABC Cafe                90.0            REVIEW                                                             Duplicate Business
    City Hospital               100.0             VALID                                                                      No Issues
  Mysore Pharmacy               100.0             VALID                                                                      No Issues
 Mangalore Bakery               100.0             VALID                                                                      No Issues
Bengaluru Library               100.0             VALID                                                                      No Issues
     XYZ Pharmacy                50.0            REVIEW   Invalid PIN Code, Missing Latitude, Missing Longitude, Missing Opening Hours
         ABC Cafe              

In [31]:
# Save the final validation report

output_file = "data/validation_report.xlsx"

df.to_excel(output_file, index=False)

print("Validation report created successfully!")
print("Saved as:", output_file)

Validation report created successfully!
Saved as: data/validation_report.xlsx


In [34]:
df = pd.read_excel(r"D:\India Maps Business Validation\Data\business_data_solution.xlsx")

In [35]:
print("Total records:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Total records: 7

Columns:
['Business Name', 'Category', 'City', 'State', 'PIN Code', 'Latitude', 'Longitude', 'Opening Hours']

First 5 records:


,Business Name,Category,City,State,PIN Code,Latitude,Longitude,Opening Hours
0,ABC Cafe,Cafe,Bengaluru,Karnataka,560001,12.9716,77.5946,9 AM - 10 PM
1,City Hospital,Hospital,Mangaluru,Karnataka,575001,12.9141,74.8560,24 Hours
2,Mysore Pharmacy,Pharmacy,Mysuru,Karnataka,570001,12.2958,76.6394,8 AM - 9 PM
3,Mangalore Bakery,Bakery,Mangaluru,Karnataka,575001,12.9145,74.8564,8 AM - 8 PM
4,Bengaluru Library,Library,Bengaluru,Karnataka,560034,12.9352,77.6245,10 AM - 6 PM


In [36]:
# Check for duplicate business records

duplicates = df[df.duplicated(subset=["Business Name", "City"], keep=False)]

print("DUPLICATE RECORDS")
print("=" * 40)

if len(duplicates) > 0:
    display(duplicates)
else:
    print("No duplicate records found.")

print("\nNumber of duplicate records:", len(duplicates))

DUPLICATE RECORDS


,Business Name,Category,City,State,PIN Code,Latitude,Longitude,Opening Hours
0,ABC Cafe,Cafe,Bengaluru,Karnataka,560001,12.9716,77.5946,9 AM - 10 PM
6,ABC Cafe,Cafe,Bengaluru,Karnataka,560001,NaN,NaN,NaN



Number of duplicate records: 2


In [37]:
# Check for missing values

print("MISSING VALUES")
print("=" * 40)

missing_values = df[df.isnull().any(axis=1)]

print("Records with missing values:", len(missing_values))

if len(missing_values) > 0:
    display(missing_values)
else:
    print("No missing values found.")

MISSING VALUES
Records with missing values: 1


,Business Name,Category,City,State,PIN Code,Latitude,Longitude,Opening Hours
6,ABC Cafe,Cafe,Bengaluru,Karnataka,560001,NaN,NaN,NaN


In [38]:
# Validate Indian PIN codes

print("PIN CODE VALIDATION")
print("=" * 40)

df["PIN Code"] = df["PIN Code"].astype(str).str.strip()

invalid_pins = df[
    ~df["PIN Code"].str.match(r"^[1-9][0-9]{5}$", na=False)
]

print("Invalid PIN code records:", len(invalid_pins))

if len(invalid_pins) > 0:
    display(invalid_pins)
else:
    print("All PIN codes are valid.")

PIN CODE VALIDATION
Invalid PIN code records: 0
All PIN codes are valid.


In [39]:
# Validate latitude and longitude

print("COORDINATE VALIDATION")
print("=" * 40)

invalid_coordinates = df[
    (df["Latitude"].isna()) |
    (df["Longitude"].isna()) |
    (df["Latitude"].abs() > 90) |
    (df["Longitude"].abs() > 180)
]

print("Invalid coordinate records:", len(invalid_coordinates))

if len(invalid_coordinates) > 0:
    display(invalid_coordinates)
else:
    print("All coordinates are valid.")

COORDINATE VALIDATION
Invalid coordinate records: 1


,Business Name,Category,City,State,PIN Code,Latitude,Longitude,Opening Hours
6,ABC Cafe,Cafe,Bengaluru,Karnataka,560001,NaN,NaN,NaN


In [40]:
# Calculate Data Quality Score

print("DATA QUALITY ANALYSIS")
print("=" * 50)

# Make sure PIN codes are strings
df["PIN Code"] = df["PIN Code"].astype(str).str.strip()

# Check duplicate records
duplicate_flag = df.duplicated(
    subset=["Business Name", "City"],
    keep=False
)

# Check invalid PIN codes
invalid_pin_flag = ~df["PIN Code"].str.match(
    r"^[1-9][0-9]{5}$",
    na=False
)

# Check invalid coordinates
invalid_coordinate_flag = (
    df["Latitude"].isna() |
    df["Longitude"].isna() |
    (df["Latitude"].abs() > 90) |
    (df["Longitude"].abs() > 180)
)

# Count missing required fields
required_columns = [
    "Business Name",
    "Category",
    "City",
    "State",
    "PIN Code",
    "Latitude",
    "Longitude",
    "Opening Hours"
]

missing_count = df[required_columns].isna().sum(axis=1)

# Calculate score
df["Data Quality Score"] = 100

# Deduct points for each issue
df["Data Quality Score"] -= missing_count * 20
df["Data Quality Score"] -= invalid_pin_flag.astype(int) * 20
df["Data Quality Score"] -= invalid_coordinate_flag.astype(int) * 20
df["Data Quality Score"] -= duplicate_flag.astype(int) * 20

# Keep score between 0 and 100
df["Data Quality Score"] = df["Data Quality Score"].clip(0, 100)

# Validation status
df["Validation Status"] = df["Data Quality Score"].apply(
    lambda score: "VALID" if score == 100 else "REVIEW"
)

print("\nDATA QUALITY RESULTS")
print("=" * 70)

display(
    df[
        [
            "Business Name",
            "Data Quality Score",
            "Validation Status"
        ]
    ]
)

DATA QUALITY ANALYSIS

DATA QUALITY RESULTS


,Business Name,Data Quality Score,Validation Status
0,ABC Cafe,80,REVIEW
1,City Hospital,100,VALID
2,Mysore Pharmacy,100,VALID
3,Mangalore Bakery,100,VALID
4,Bengaluru Library,100,VALID
5,Sai Medical and Polyclinic,100,VALID
6,ABC Cafe,0,REVIEW


In [41]:
# Identify the specific issues in each business record

def find_issues(row):
    issues = []

    # Duplicate check
    if df.duplicated(
        subset=["Business Name", "City"],
        keep=False
    ).loc[row.name]:
        issues.append("Duplicate Record")

    # Missing values
    if pd.isna(row["Latitude"]):
        issues.append("Missing Latitude")

    if pd.isna(row["Longitude"]):
        issues.append("Missing Longitude")

    if pd.isna(row["Opening Hours"]):
        issues.append("Missing Opening Hours")

    # PIN validation
    pin = str(row["PIN Code"]).strip()

    if not pd.Series([pin]).str.match(
        r"^[1-9][0-9]{5}$"
    ).iloc[0]:
        issues.append("Invalid PIN Code")

    # Coordinate validation
    if (
        pd.notna(row["Latitude"])
        and abs(row["Latitude"]) > 90
    ):
        issues.append("Invalid Latitude")

    if (
        pd.notna(row["Longitude"])
        and abs(row["Longitude"]) > 180
    ):
        issues.append("Invalid Longitude")

    if len(issues) == 0:
        return "No Issues"

    return ", ".join(issues)


df["Issues Found"] = df.apply(find_issues, axis=1)

print("FINAL VALIDATION REPORT")
print("=" * 100)

display(
    df[
        [
            "Business Name",
            "Data Quality Score",
            "Validation Status",
            "Issues Found"
        ]
    ]
)

FINAL VALIDATION REPORT


,Business Name,Data Quality Score,Validation Status,Issues Found
0,ABC Cafe,80,REVIEW,Duplicate Record
1,City Hospital,100,VALID,No Issues
2,Mysore Pharmacy,100,VALID,No Issues
3,Mangalore Bakery,100,VALID,No Issues
4,Bengaluru Library,100,VALID,No Issues
5,Sai Medical and Polyclinic,100,VALID,No Issues
6,ABC Cafe,0,REVIEW,"Duplicate Record, Missing Latitude, Missing Lo..."


In [42]:
# Save the final validation report

output_file = "data/final_validation_report.xlsx"

df.to_excel(output_file, index=False)

print("Final validation report created successfully!")
print("Saved as:", output_file)

Final validation report created successfully!
Saved as: data/final_validation_report.xlsx
